#### Imports: orjson to read jsons faster, NetworkX to build graphs, pickle to load into memory faster, defaultdict to speed up dictionary creation


In [1]:
import pandas as pd
import orjson
import networkx as nx
import pickle
from collections import defaultdict
import hashlib
from tqdm import tqdm
from pyvis.network import Network

#### Load into Dataframe

In [2]:
def load_jsonl(filepath):
    with open(filepath, "rb") as f:
        return pd.DataFrame([orjson.loads(line) for line in f])
comments_df = load_jsonl("r_ApplyingToCollege_comments.jsonl")

print(f"Comments: {len(comments_df):,}")
print(comments_df.columns.tolist())

KeyboardInterrupt: 

In [8]:
list(posts_df.columns)

['_meta',
 'all_awardings',
 'allow_live_comments',
 'approved_at_utc',
 'approved_by',
 'archived',
 'author',
 'author_flair_background_color',
 'author_flair_css_class',
 'author_flair_richtext',
 'author_flair_template_id',
 'author_flair_text',
 'author_flair_text_color',
 'author_flair_type',
 'author_fullname',
 'author_is_blocked',
 'author_patreon_flair',
 'author_premium',
 'awarders',
 'banned_at_utc',
 'banned_by',
 'can_gild',
 'can_mod_post',
 'category',
 'clicked',
 'content_categories',
 'contest_mode',
 'created',
 'created_utc',
 'discussion_type',
 'distinguished',
 'domain',
 'downs',
 'edited',
 'gilded',
 'gildings',
 'hidden',
 'hide_score',
 'id',
 'is_created_from_ads_ui',
 'is_crosspostable',
 'is_meta',
 'is_original_content',
 'is_reddit_media_domain',
 'is_robot_indexable',
 'is_self',
 'is_video',
 'likes',
 'link_flair_background_color',
 'link_flair_css_class',
 'link_flair_richtext',
 'link_flair_template_id',
 'link_flair_text',
 'link_flair_text_colo

In [11]:
posts_df2 = posts_df[["created_utc", "selftext", "title"]].copy()

In [12]:
posts_df2.tail()

,created_utc,selftext,title
246963,1778999976,Give me best mpc campus and college for mpc fo...,Best inter college for mpc
246964,1779000307,From the recent 2021 U.S. Department of Educat...,Who colleges believe are their peers?
246965,1779000633,"Hey,\n\nI am being listening about placements ...",Reality about placements in Thapar University
246966,1779000659,[removed],How many ppl do yall think got in through USC ...
246967,1779001280,"Basically the title. As a sophomore, this year...",How do you get closer to your teachers?


#### Bot Filter

In [3]:
bots = {"[deleted]", "AutoModerator", "RemindMeBot", "sneakpeek_bot","ApplyingToCollege-ModTeam"}
comments_df = comments_df[~comments_df["author"].isin(bots)]
comments_df = comments_df[comments_df["author"].notna()]
print(f"After bot filter: {len(comments_df):,}")

After bot filter: 1,833,446


#### Anonymize

In [5]:
def anonymize(username):
    return "user_" + hashlib.md5(username.encode()).hexdigest()[:8]

comments_df["author"] = comments_df["author"].apply(anonymize)


#### Time Filter

In [6]:
comments_df["created_utc"] =pd.to_numeric(comments_df["created_utc"],errors="coerce")
comments_df = comments_df[
 (comments_df["created_utc"] >= 1704067200) & (comments_df["created_utc"] <  1756684800) #2024 to 2025 September   
]
print(f"After Time Filter:{len(comments_df):,}")

After Time Filter:1,297,402


#### Active User Filter

In [7]:
user_counts = comments_df["author"].value_counts()
active_users = set(user_counts[user_counts >= 10].index)
comments_df = comments_df[comments_df["author"].isin(active_users)]
print(f"Active users: {len(active_users):,}")
print(f"After user filter: {len(comments_df):,}")


Active users: 19,447
After user filter: 1,037,717


#### Author Lookup

In [8]:
author_lookup = dict(zip(comments_df["id"], comments_df["author"]))

#### Edge List, and User Activity

In [11]:
edge_weights = defaultdict(int)
user_activity = defaultdict(lambda: {
    "comment_count": 0,
    "total_score": 0,
    "controversial_count": 0,
    "posts_active_in": set()
})

for _, row in tqdm(comments_df.iterrows(), total=len(comments_df)):
    author = row["author"]
    parent_id = str(row["parent_id"])
    link_id = str(row["link_id"])

    ua = user_activity[author]
    ua["comment_count"] += 1
    ua["total_score"] += row["score"] or 0
    ua["posts_active_in"].add(link_id)
    if row.get("controversiality", 0):
        ua["controversial_count"] += 1

    if parent_id.startswith("t1_"):
        parent_comment_id = parent_id[3:]
        parent_author = author_lookup.get(parent_comment_id)
        if parent_author and parent_author != author:
            edge_weights[(author, parent_author)] += 1

100%|██████████| 1037717/1037717 [00:13<00:00, 75553.88it/s]


#### Weak Edge Filter and Convert to dictionary

In [12]:
edge_weights = {k: v for k, v in edge_weights.items() if v >= 3}
user_activity = dict(user_activity)
print(f"Edges after filter: {len(edge_weights):,}")

Edges after filter: 24,866


#### Building a Network

In [13]:
G = nx.DiGraph()

for (src, tgt), weight in edge_weights.items():
    G.add_edge(src, tgt, weight=weight)

for user, stats in user_activity.items():
    if G.has_node(user):
        G.nodes[user].update({
            "comment_count": stats["comment_count"],
            "total_score": stats["total_score"],
            "controversial_count": stats["controversial_count"],
            "posts_active": len(stats["posts_active_in"]),
        })

print(f"Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")

Graph: 9,605 nodes, 24,866 edges


#### Computing metrics

In [14]:
# Betweenness reads `weight` as a COST (longer = further apart), but our weights are
# interaction counts (higher = closer). Invert them into a distance so shortest paths
# actually route through strong ties.
for _, _, d in G.edges(data=True):
    d["distance"] = 1 / d["weight"]

pagerank = nx.pagerank(G, weight="weight")

betweenness = nx.betweenness_centrality(G, weight="distance", k=300, seed=42)

in_degree  = dict(G.in_degree(weight="weight"))
out_degree = dict(G.out_degree(weight="weight"))

#### New Dataframe for Metrics

In [15]:
nodes = list(G.nodes())
metrics_df = pd.DataFrame({
    "user":                [n for n in nodes],
    "pagerank":            [pagerank.get(n, 0) for n in nodes],
    "replies_received":    [in_degree.get(n, 0) for n in nodes],
    "replies_given":       [out_degree.get(n, 0) for n in nodes],
    "betweenness":         [betweenness.get(n, 0) for n in nodes],
    "comment_count":       [G.nodes[n].get("comment_count", 0) for n in nodes],
    "total_score":         [G.nodes[n].get("total_score", 0) for n in nodes],
    "controversial_count": [G.nodes[n].get("controversial_count", 0) for n in nodes],
    "posts_active":        [G.nodes[n].get("posts_active", 0) for n in nodes],
})

metrics_df["influence_score"] = (
    metrics_df["pagerank"] * 1000 * 0.35 +
    metrics_df["replies_received"] * 0.25 +
    metrics_df["betweenness"] * 100 * 0.20 +
    metrics_df["total_score"].clip(0) * 0.001 * 0.20
)

def classify(row):
    controversial_rate = row["controversial_count"] / max(row["comment_count"], 1)
    is_debater = controversial_rate > 0.01 and row["controversial_count"] > 10
    is_bridge = row["betweenness"] > metrics_df["betweenness"].quantile(0.92)
    
    if is_bridge and is_debater:
        return "Bridge-Debater"  # both
    if is_debater:
        return "Debater"
    if is_bridge:
        return "Bridge"
    return "Advisor"

metrics_df["role"] = metrics_df.apply(classify, axis=1)
metrics_df = metrics_df.sort_values("influence_score", ascending=False).reset_index(drop=True)

print("\nTop 10 most influential users:")
print(metrics_df[["user", "role", "influence_score", "comment_count", "total_score"]].head(10))


Top 10 most influential users:
            user            role  influence_score  comment_count  total_score
0  user_e6ee5a63  Bridge-Debater      2531.603379          34411       179633
1  user_5875a538          Bridge       811.131953          18316       108182
2  user_2896a9a8          Bridge       384.933532           8664        48800
3  user_65474719          Bridge       358.905523           7831        48644
4  user_268205c4  Bridge-Debater       332.852423           5083        21074
5  user_65488c37          Bridge       302.215291          15297        39420
6  user_4b04608a  Bridge-Debater       295.513792           4502        31150
7  user_a53a4265          Bridge       277.484955           7066        56316
8  user_fb4c9fec          Bridge       209.085529           5025        31789
9  user_fa796138          Bridge       200.837357           6335        23912


#### Pickling NetwrokX Graph and Metrics DF 

In [16]:
with open("graph.pkl", "wb") as f:
    pickle.dump(G, f)

with open("metrics.pkl", "wb") as f:
    pickle.dump(metrics_df, f)


#### Opening pickled objects

In [3]:
with open("graph.pkl", "rb") as f:
    G = pickle.load(f)

with open("metrics.pkl", "rb") as f:
    metrics_df = pickle.load(f)

#### Comment-Text Cache (for TF-IDF Cluster Keywords)
Builds `user_word_counts.pkl` once by streaming the raw comment archive; every later re-run at a different `RESOLUTION` reuses it instantly, since cluster membership changes but each user's own vocabulary doesn't.

In [4]:
import os, time, re, collections, orjson

WORD_CACHE = "user_word_counts.pkl"

# Per-user comment vocabulary, used below to label clusters with real TF-IDF
# keywords. Cached to disk since scanning the 3.4GB comment archive is
# unnecessary every time you re-run the visualization at a different
# RESOLUTION — cluster membership changes, but each user's own vocabulary
# doesn't, so this only needs to run once.
if os.path.exists(WORD_CACHE):
    with open(WORD_CACHE, "rb") as f:
        user_word_counts = pickle.load(f)
    print(f"loaded {WORD_CACHE}: {len(user_word_counts):,} users")
else:
    graph_nodes = set(G.nodes())

    # graph.pkl's node ids are DOUBLE hashed — anonymize() above was applied to
    # comments_df twice in the kernel that produced this pickle. Mirror that
    # here so raw usernames resolve to the same ids used everywhere else in
    # this notebook. If you ever rebuild graph.pkl from a clean top-to-bottom
    # run, drop the second call.
    def _double_anon(u): return anonymize(anonymize(u))

    STOP_TOKENS = set("""
    the a an and or but if then so because as of to in on for with at by from
    up down out over under again further this that these those is are was
    were be been being have has had do does did will would should could can
    may might must shall not no nor you your yours yourself i me my mine we
    us our ours they them their theirs he him his she her hers it its im ive
    dont cant wont didnt doesnt isnt wasnt arent werent youre youve theyre
    theyve whats thats heres theres whos whens wheres whys hows lets
    """.split())
    TOKEN = re.compile(r"[a-z][a-z']{2,}")
    WINDOW = (1704067200, 1756684800)   # same window used to build the edges

    counts = collections.defaultdict(collections.Counter)
    t0 = time.time()
    with open("r_ApplyingToCollege_comments.jsonl", "rb") as f:
        for line in f:
            try:
                d = orjson.loads(line)
            except Exception:
                continue
            a = d.get("author")
            if not a:
                continue
            uid = _double_anon(a)
            if uid not in graph_nodes:
                continue
            t = d.get("created_utc")
            try:
                t = float(t)
            except (TypeError, ValueError):
                continue
            if not (WINDOW[0] <= t < WINDOW[1]):
                continue
            body = (d.get("body") or "").lower()
            c = counts[uid]
            for w in TOKEN.findall(body):
                if w not in STOP_TOKENS:
                    c[w] += 1

    user_word_counts = dict(counts)
    print(f"built vocabulary for {len(user_word_counts):,} of {len(graph_nodes):,} graph users "
          f"in {time.time() - t0:.0f}s")
    with open(WORD_CACHE, "wb") as f:
        pickle.dump(user_word_counts, f)

loaded user_word_counts.pkl: 9,605 users


#### Pyvis Graph


In [8]:
import math
import collections
import networkx as nx
from pyvis.network import Network

# ── Select by COMMUNITY, not by influence rank ─────────────────────
# Ranking by influence selects hubs, and hubs are precisely the users who span
# communities rather than sit inside one — so a top-N-by-influence slice has no
# cluster structure left to draw (Q = 0.23 at N=300). Partition first, then
# sample the dense core of each community.
#
# RESOLUTION tunes how tight the communities are. It is a real model parameter,
# not a fudge factor: modularity has a known resolution limit (Fortunato &
# Barthelemy 2007) so gamma=1 is NOT a neutral default — it merges genuinely
# distinct small groups. Picking it right requires three checks, not just one:
#   (a) partition stability — mean ARI across 6 Louvain seeds, higher = the
#       grouping isn't an artifact of which random seed you happened to run
#   (b) community count (size>=30) — this PEAKS around gamma=6-6.5 (139) then
#       DECLINES (115 by gamma=10) even as ARI keeps climbing. Past the peak,
#       higher resolution isn't finding more structure, it's just collapsing
#       an already-shrinking set of communities into a more rigid shape —
#       stability without substance.
#   (c) topic recoverability — do the TF-IDF keywords (below) actually read as
#       real topics? This is the strongest test, since it checks against
#       comment text the clustering never saw. It also peaks at gamma=6-6.5:
#       gamma=6.0 recovers 5 of 8 clusters as distinct real topics (including
#       a "prestige privates" group — UVA/Georgetown/Duke/Dartmouth — that
#       doesn't appear at any other gamma). By gamma=7+ that collapses to 2,
#       replaced by generic filler ("such, sort, fact, sometimes"), even
#       though ARI is still rising.
# Swept gamma=3..10 in steps of 0.5 on this graph:
#   1.0 -> 1767 edges, 41% intra, ARI 0.48   <- least reproducible setting
#   3.0 ->  704 edges, 63% intra, ARI 0.74,  104 communities, 2/8 topical
#   5.0 ->  573 edges, 68% intra, ARI 0.81,  138 communities, 4/8 topical
#   6.0 ->  ARI 0.811 (mean) / 0.797 (min),  139 communities (peak), 5/8 topical  <- best on all three checks
#   6.5 ->  ARI 0.820,                       139 communities (tied peak), 3/8 topical
#   8.0 ->  ARI 0.848,                       128 communities (falling), 2/8 topical
#  10.0 ->  ARI 0.860 (highest),             115 communities (falling), 2/8 topical
# Whatever you pick, REPORT it: these are "communities at resolution gamma",
# never "the" communities.
RESOLUTION    = 6
N_COMMUNITIES = 10
PEEL_TO       = 200      # nodes kept per community

UG = G.to_undirected()
communities = nx.community.louvain_communities(
    UG, weight="weight", seed=42, resolution=RESOLUTION
)
# Louvain returns SETS, and Python randomises string hashes per process, so set
# iteration order changes every run. Sort each community and order them
# explicitly — otherwise the peel below takes a different path each time and you
# get a different 360-node sample (and a different-looking picture) on every run.
# `seed=42` only controls Louvain's own RNG, not iteration order.
communities = sorted((sorted(c) for c in communities if len(c) >= 30),
                     key=lambda c: (-len(c), c[0]))

# Keep each community's dense core: repeatedly drop its weakest-connected member.
# Taking the top-N by degree instead would re-select the boundary-spanning hubs.
node_comm = {}
for ci, comm in enumerate(communities[:N_COMMUNITIES]):
    sub = UG.subgraph(comm).copy()
    while sub.number_of_nodes() > PEEL_TO:
        # tie-break on the name so the victim is well-defined
        sub.remove_node(min(sub.nodes(), key=lambda n: (sub.degree(n, weight="weight"), n)))
    for n in sub.nodes():
        node_comm[n] = ci

# Build the subgraph explicitly rather than G.subgraph(...).copy(): that returns
# a VIEW whose node filter holds the induced nodes in a set, and for a small
# induced set the view iterates that set — so node order (and therefore the
# order everything downstream sees) is hash-randomised per process.
_view = G.subgraph(node_comm.keys())
H = nx.DiGraph()
H.add_nodes_from((n, G.nodes[n]) for n in sorted(node_comm))
for u, v in sorted(_view.edges()):
    H.add_edge(u, v, **_view.edges[u, v])
H.remove_nodes_from(sorted(nx.isolates(H)))

_U = H.to_undirected()
_intra = sum(1 for a, b in _U.edges() if node_comm[a] == node_comm[b])
print(f"{min(N_COMMUNITIES, len(communities))} communities | "
      f"{H.number_of_nodes()} nodes | {_U.number_of_edges()} edges | "
      f"{_intra / _U.number_of_edges():.1%} of edges intra-community")

pr     = metrics_df.set_index("user")["pagerank"].to_dict()
roles  = metrics_df.set_index("user")["role"].to_dict()
max_pr = max(pr[n] for n in H.nodes())

# one hue per community, tuned for the dark ground
comm_colors = ["#7c6af7", "#00d4aa", "#e84393", "#ffa94d",
               "#4dabf7", "#c0eb75", "#ffd43b", "#ff8787",
               "#39A57A", "#eb88db"]

role_colors = {
    "Advisor":        "#7c6af7",
    "Debater":        "#e84393",
    "Bridge":         "#00d4aa",
    "Bridge-Debater": "#ffffff",
}

PHYSICS = """
{
  "physics": {
    "stabilization": {
      "enabled": true,
      "iterations": 800,
      "updateInterval": 50
    },
    "minVelocity": 0.75,
    "maxVelocity": 50,
    "barnesHut": {
      "gravity": -2500,
      "centralGravity": 0.1,
      "springLength": 150,
      "springConstant": 0.06,
      "damping": 0.9,
      "avoidOverlap": 0.2
    }
  }
}
"""



# ── Cluster keyword legend (TF-IDF) ─────────────────────────────────
# Each cluster's pooled comments treated as one "document":
#   tf  = how often a word appears within this cluster
#   idf = how few OTHER clusters also use it — smoothed (sklearn convention:
#         log((1+N)/(1+df)) + 1) so a word used in every cluster is
#         down-weighted rather than deleted. With only 8 "documents", plain
#         idf goes negative for anything universal and silently drops real
#         signal — e.g. "international"/"financial aid" are genuinely
#         distinctive but appear in all 8 clusters at some level, so raw idf
#         zeroes them out entirely.
# Two guards keep this from surfacing garbage: a word must be used by several
# DISTINCT members of a cluster (one prolific poster's private joke or a
# garbled token shouldn't count), and a DATA-DRIVEN stopword list removes the
# words used by nearly everyone in this specific subreddit — "school",
# "college", "gpa" carry no information about WHICH cluster someone is in,
# same as "the" or "and", but a generic English stopword list wouldn't know
# that about this particular corpus.
_global_freq = collections.Counter()
for _c in user_word_counts.values():
    _global_freq.update(_c)
_STOP = {w for w, _ in _global_freq.most_common(180)}

# Vocabulary is pooled from the FULL community (communities[:N_COMMUNITIES]),
# not just the PEEL_TO nodes actually drawn — peeling is a rendering choice,
# and 45 nodes is thin enough that one heavy poster can dominate a cluster's
# apparent vocabulary.
_cluster_counts = [collections.Counter() for _ in communities[:N_COMMUNITIES]]
_cluster_userdf = [collections.Counter() for _ in communities[:N_COMMUNITIES]]
for _ci, _comm in enumerate(communities[:N_COMMUNITIES]):
    for _n in _comm:
        _c = user_word_counts.get(_n)
        if not _c:
            continue
        _cc = {w: v for w, v in _c.items() if w not in _STOP}
        _cluster_counts[_ci].update(_cc)
        _cluster_userdf[_ci].update(_cc.keys())

_n_clusters = len(_cluster_counts)
_totals = [sum(c.values()) for c in _cluster_counts]
_vocab = set()
for _c in _cluster_counts:
    _vocab.update(_c)
_corpus_freq = {w: sum(c[w] for c in _cluster_counts) for w in _vocab}
_doc_freq    = {w: sum(1 for c in _cluster_counts if c[w] > 0) for w in _vocab}


def _keywords(ci, k=6, min_users=4, min_corpus=15):
    c, total, udf = _cluster_counts[ci], _totals[ci], _cluster_userdf[ci]
    scored = []
    for w, cnt in c.items():
        if udf[w] < min_users or _corpus_freq[w] < min_corpus:
            continue
        tf  = cnt / total
        idf = math.log((1 + _n_clusters) / (1 + _doc_freq[w])) + 1
        scored.append((tf * idf, w))
    scored.sort(reverse=True)
    return [w for _, w in scored[:k]]


cluster_keywords = {ci: _keywords(ci) for ci in range(_n_clusters)}
for ci, kws in cluster_keywords.items():
    print(f"  cluster {ci}: {', '.join(kws) if kws else '(no distinctive terms above threshold)'}")


def _legend_html():
    rows = "".join(
        f'<div style="display:flex;align-items:baseline;gap:8px;margin:4px 0;">'
        f'<span style="width:9px;height:9px;border-radius:50%;flex:none;'
        f'background:{comm_colors[ci % len(comm_colors)]};"></span>'
        f'<span><b style="opacity:.9">{ci}</b> — '
        f'{", ".join(cluster_keywords.get(ci, [])) or "no distinctive terms"}</span>'
        f'</div>'
        for ci in sorted(cluster_keywords)
    )
    return (
        '<div style="position:fixed;top:14px;right:14px;max-width:360px;'
        'background:rgba(10,10,15,.90);border:1px solid #322f47;border-radius:8px;'
        'padding:12px 15px;font:12px/1.55 -apple-system,Segoe UI,Roboto,sans-serif;'
        'color:#e8e8f0;z-index:1000;">'
        '<div style="font-size:10px;font-weight:600;letter-spacing:.05em;'
        'text-transform:uppercase;opacity:.55;margin-bottom:7px;">'
        'Cluster keywords (TF-IDF)</div>'
        f"{rows}</div>"
    )


def _inject_legend(filename):
    with open(filename, "r", encoding="utf-8") as f:
        html = f.read()
    html = html.replace("<body>", "<body>\n" + _legend_html(), 1)
    with open(filename, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"legend injected into {filename}")


def render(color_by, filename):
    """Draw H twice over the same layout, changing only the node colour channel.
    color_by: "community" or "role"."""
    net = Network(
        height="800px",
        width="100%",
        directed=False,
        bgcolor="#0a0a0f",
        font_color="#e8e8f0"
    )

    for node in H.nodes():
        ci   = node_comm[node]
        role = roles.get(node, "Advisor")
        p    = pr.get(node, 0)
        nd   = G.nodes[node]

        if color_by == "community":
            color = comm_colors[ci % len(comm_colors)]
        else:
            color = role_colors.get(role, "#7c6af7")

        tip = (
            f"{node},  "
            f"Cluster: {ci},  "
            f"Role: {role},  "
            f"Influence: {p:.4f},  "
            f"Comments: {nd.get('comment_count', 0)},  "
            f"Score: {nd.get('total_score', 0)},  "
            f"Controversial: {nd.get('controversial_count', 0)}"
        )
        net.add_node(
            node,
            label=" ",
            size=10 + (p / max_pr) * 25,
            color=color,
            title=tip,
        )

    # ── Edges: tie strength drives the LAYOUT, not just the line width ──
    # vis-network ignores `value` for physics — it only scales rendered width,
    # which is why weight had no effect on the picture before. `length` is the
    # spring rest length, so a strong tie must map to a SHORT spring. Weights
    # are heavily skewed (median 3, max 270), so compress on a log scale or
    # ~75% of edges end up looking alike.
    max_w   = max(d["weight"] for _, _, d in H.edges(data=True))
    log_max = math.log1p(max_w)

    for src, tgt, data in H.edges(data=True):
        w        = data["weight"]
        strength = math.log1p(w) / log_max                  # 0..1
        alpha    = int(25 + strength * 128)
        net.add_edge(
            src, tgt,
            value=w,
            length=300 - strength * 250,                    # weak 300 → strong 50
            color=f"#ffffff{alpha:02x}",
        )

    net.set_options(PHYSICS)
    net.save_graph(filename)
    print(f"wrote {filename}  (coloured by {color_by})")


render("community", "network_clusters.html")
render("role",      "network_roles.html")

# Legend only makes sense on the community-coloured file — role colouring
# doesn't correspond to these cluster ids.
_inject_legend("network_clusters.html")

10 communities | 1230 nodes | 1582 edges | 80.2% of edges intra-community
  cluster 0: applicant, extraordinarily, scholarships, specific, attend, ability
  cluster 1: courses, admission, ucla, csu, edu, cal
  cluster 2: i'd, study, attend, selective, admit, pick
  cluster 3: check, you'll, won't, submit, looking, option
  cluster 4: caltech, stanford, princeton, please, ivies, went
  cluster 5: alabama, edu, team, news, national, scholarships
  cluster 6: such, sort, fact, least, sometimes, selective
  cluster 7: uva, georgetown, prestige, duke, dartmouth, ucla
  cluster 8: transfer, california, ucla, admission, berkeley, ucs
  cluster 9: transfer, won't, waitlist, scholarships, put, there's
wrote network_clusters.html  (coloured by community)
wrote network_roles.html  (coloured by role)
legend injected into network_clusters.html


In [74]:
metrics_df["role"] = metrics_df.apply(classify, axis=1)
print(metrics_df["role"].value_counts())

role
Advisor           8150
Bridge            1333
Bridge-Debater     108
Debater             14
Name: count, dtype: int64


In [84]:
test = dict(edge_weights)